<a href="https://colab.research.google.com/github/Himanshu0508Raturi/Fine-Tuning-FLAN-T5/blob/main/DistilBERT_SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
import evaluate

# Detect device automatically — works on CPU, single GPU, multi-GPU via Accelerate
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [3]:
# Centralise all hyperparameters so you can tune them in one place.
MODEL_NAME       = "distilbert-base-uncased"   # base checkpoint
DATASET_NAME     = "stanfordnlp/sst2"          # Hugging Face dataset ID
NUM_LABELS       = 2                            # positive / negative
MAX_LENGTH       = 128                          # tokeniser truncation
BATCH_SIZE       = 64                           # per-device train batch size
LEARNING_RATE    = 5e-5
NUM_EPOCHS       = 3
FREEZE_BACKBONE  = True                         # freeze DistilBERT, train classifier head only
OUTPUT_DIR       = "./distilbert-sst2"          # checkpoints + final model

In [4]:
## 4. Load the SST-2 Dataset
dataset = load_dataset(DATASET_NAME)
print(dataset)
print("\nSample train example:")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

Sample train example:
{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}


In [5]:
## Tokenise the Dataset - We return plain dicts (no `return_tensors`) — `DataCollatorWithPadding` handles dynamic padding per batch, which is more memory-efficient than padding everything to `max_length` upfront.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
        # No padding here — DataCollatorWithPadding handles it dynamically
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["sentence", "idx"],  # drop columns Trainer does not need
)
print(tokenized_datasets)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})


In [6]:
## 6. Data Collator - `DataCollatorWithPadding` pads each batch to the length of its longest sequence, reducing wasted compute compared to global padding.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Sanity check — inspect a batch shape
sample_batch = data_collator([tokenized_datasets["train"][i] for i in range(4)])
print({k: v.shape for k, v in sample_batch.items()})

{'input_ids': torch.Size([4, 15]), 'token_type_ids': torch.Size([4, 15]), 'attention_mask': torch.Size([4, 15]), 'labels': torch.Size([4])}


In [7]:
## 7. Load DistilBERT and Optionally Freeze the Backbone -
'''We load `distilbert-base-uncased` with a randomly initialised 2-class classification head.
Setting `FREEZE_BACKBONE = True` replicates the original notebook's intent — training only
the top layer is faster, requires less data, and is less prone to catastrophic forgetting.'''
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

if FREEZE_BACKBONE:
    # Freeze all DistilBERT transformer parameters; keep classifier head trainable
    for param in model.distilbert.parameters():
        param.requires_grad_(False)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
else:
    total = sum(p.numel() for p in model.parameters())
    print(f"Full fine-tune — all {total:,} parameters trainable")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 592,130 / 66,955,010 (0.9%)


In [8]:
## 8. Evaluation Metrics

'''The `evaluate` library (the modern successor to the deprecated `datasets.load_metric` API)
provides standard, reproducible implementations of accuracy and F1.'''
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="binary")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}


In [9]:
## 9. Training Arguments

'''`TrainingArguments` replaces the manual `.compile()` call and controls the entire training loop:
mixed-precision, gradient accumulation, logging, evaluation scheduling, and checkpointing.'''
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training schedule
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    # Optimiser
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,              # L2 regularisation on non-bias params
    warmup_ratio=0.06,              # linear warmup over first 6% of steps

    # Mixed precision (automatic)
    fp16=torch.cuda.is_available(), # fp16 on GPU, full precision on CPU

    # Evaluation & checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    # Logging
    logging_steps=100,
    report_to="none",              # change to "wandb" or "tensorboard" if desired
    push_to_hub=False,             # set True + hub_model_id to publish to HF Hub
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
## 10. Initialise the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
## 11. Train the Model
train_result = trainer.train()

# Log training summary
print("\n Training Summary")
print(f"Total steps     : {train_result.global_step}")
print(f"Training loss   : {train_result.training_loss:.4f}")
print(f"Runtime         : {train_result.metrics['train_runtime']:.1f}s")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.377035,0.383420,0.830275,0.831050
2,0.377450,0.375538,0.836009,0.836944
3,0.361055,0.374793,0.833716,0.835787


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



 Training Summary
Total steps     : 3159
Training loss   : 0.4017
Runtime         : 98.4s


In [13]:
## 12. Evaluate on the Validation Set
eval_results = trainer.evaluate()

print("\n Evaluation Results ")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


 Evaluation Results 
  eval_loss: 0.3755
  eval_accuracy: 0.8360
  eval_f1: 0.8369
  eval_runtime: 0.4586
  eval_samples_per_second: 1901.5670
  eval_steps_per_second: 30.5300
  epoch: 3.0000


In [14]:
## 13. Save the Fine-Tuned Model

'''`save_pretrained` stores both weights and tokeniser config so the model can be reloaded with a single `from_pretrained` call.'''
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")

# List saved files
import os
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s}  {size/1e6:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./distilbert-sst2/
  checkpoint-1053                           0.0 MB
  checkpoint-2106                           0.0 MB
  checkpoint-3159                           0.0 MB
  config.json                               0.0 MB
  model.safetensors                         267.8 MB
  tokenizer.json                            0.7 MB
  tokenizer_config.json                     0.0 MB
  training_args.bin                         0.0 MB


In [17]:
# Testing
# Load directly from the saved directory
classifier = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

test_sentences = [
    "The movie was absolutely brilliant and left me in tears of joy.",
    "A complete waste of two hours. The plot made no sense whatsoever.",
    "Not bad, but the second half dragged on a bit.",
    "One of the finest performances I have seen in recent memory.",
    "Very Bad",
]

results = classifier(test_sentences)

print(f"{'Sentence':60s}  {'Label':10s}  Score")
print("-" * 85)
for sentence, result in zip(test_sentences, results):
    print(f"{sentence[:58]:60s}  {result['label']:10s}  {result['score']:.4f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence                                                      Label       Score
-------------------------------------------------------------------------------------
The movie was absolutely brilliant and left me in tears of    LABEL_1     0.9868
A complete waste of two hours. The plot made no sense what    LABEL_0     0.9755
Not bad, but the second half dragged on a bit.                LABEL_0     0.5730
One of the finest performances I have seen in recent memor    LABEL_1     0.9966
Very Bad                                                      LABEL_0     0.8590


In [18]:
import shutil

folder_to_zip = "distilbert-sst2"
output_zip = "distilbert-sst2"

shutil.make_archive(output_zip, 'zip', folder_to_zip)

print("ZIP file created!")

ZIP file created!
